<a href="https://colab.research.google.com/github/rakshachahar/flyrank-ml-internship/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rakshachahar/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents the search performance of one anonymized content page for a specific client on a specific reporting date.

For this assignment, I will use a mid-panel month (March 2026) to analyze observable search-performance signals that support content refresh decisions.

In [ ]:
%pip -q install duckdb huggingface_hub pandas

import os
import duckdb
import pandas as pd

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

fact_daily = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

print("✅ Connected successfully.")

✅ Connected successfully.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features**
- Clicks
- Impressions
- CTR
- Average Position
- Date

**Label / Proxy**
- Whether a page should be prioritized for content refresh.

**Context**
- Client ID
- Content ID
- Reporting Date

**Excluded**
- Any future performance information or columns derived from future outcomes because they would introduce data leakage.

In [ ]:
# Display the selected fields for this project

features = [
    "gsc_clicks",
    "gsc_impressions",
    "gsc_ctr",
    "gsc_avg_position"
]

print("Selected Features:")
for feature in features:
    print("-", feature)

label = "Content Refresh Priority (Proxy)"

print("\nLabel / Proxy:")
print(label)

print("\nExcluded:")
print("- Future information")
print("- Any label-derived columns")

Selected Features:
- gsc_clicks
- gsc_impressions
- gsc_ctr
- gsc_avg_position

Label / Proxy:
Content Refresh Priority (Proxy)

Excluded:
- Future information
- Any label-derived columns


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The following queries verify the assumptions used in this project. They check the unit of analysis, the number of available records, the time window covered by the selected data, and whether important fields contain missing values. These checks help confirm that the dataset is suitable for building a content refresh prioritization model.

In [ ]:
# Verify the dataset with simple queries

# 1. Total number of rows
row_count = con.sql(f"""
SELECT COUNT(*) AS total_rows
FROM {fact_daily}
""").df()

print("Total Rows:")
display(row_count)

# 2. Date range
date_range = con.sql(f"""
SELECT
MIN(report_date) AS start_date,
MAX(report_date) AS end_date
FROM {fact_daily}
""").df()

print("Date Range:")
display(date_range)

# 3. Missing values
missing = con.sql(f"""
SELECT
COUNT(*) AS total_rows,
COUNT(gsc_clicks) AS clicks_available,
COUNT(gsc_impressions) AS impressions_available,
COUNT(gsc_avg_position) AS position_available
FROM {fact_daily}
""").df()

print("Column Availability:")
display(missing)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total Rows:


,total_rows
0,78835655


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Date Range:


,start_date,end_date
0,2025-01-27,2026-06-30


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Column Availability:


,total_rows,clicks_available,impressions_available,position_available
0,78835655,78737649,78737649,28970001


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset cannot prove that refreshing a page will improve rankings or traffic. It contains observed search-performance signals that support decision-making but cannot establish cause-and-effect relationships or predict Google's ranking algorithm. Some clients also have different historical coverage, so comparisons across all clients may not always be perfectly balanced.

In [ ]:
print("Data Limitation Check")

limitations = [
    "Different clients have different history lengths.",
    "The dataset contains observed search signals only.",
    "The data cannot prove causation.",
    "Recommendations are decision-support only."
]

for item in limitations:
    print("-", item)

Data Limitation Check
- Different clients have different history lengths.
- The dataset contains observed search signals only.
- The data cannot prove causation.
- Recommendations are decision-support only.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.